# MarsLandformNet V3 — Tile Classifier GPU Training

**Setup:** Runtime → Change runtime type → **GPU (T4)**

Then run all cells top-to-bottom.

In [ ]:
# Cell 1: Enter your GitHub PAT (repo scope) to download private release data
# Create one at: https://github.com/settings/tokens → Generate new token (classic) → check 'repo'
import getpass
GITHUB_TOKEN = getpass.getpass('GitHub PAT (repo scope): ')

In [ ]:
# Cell 2: Download training data from private release
import subprocess, os, json, urllib.request
from pathlib import Path

ROOT = Path('/content/marslab_v3')
ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)

TAR = ROOT / 'v3_training_data.tar.gz'
if not TAR.exists():
    req = urllib.request.Request(
        'https://api.github.com/repos/jejuchild/MarsLab/releases/tags/v3-training-data',
        headers={'Authorization': f'token {GITHUB_TOKEN}', 'Accept': 'application/vnd.github.v3+json'}
    )
    release = json.loads(urllib.request.urlopen(req).read())
    asset = release['assets'][0]
    asset_url = asset['url']
    print(f"Downloading {asset['name']} ({asset['size']/(1024*1024):.1f} MB)...")
    subprocess.check_call([
        'curl', '-sL', '-o', str(TAR),
        '-H', f'Authorization: token {GITHUB_TOKEN}',
        '-H', 'Accept: application/octet-stream',
        asset_url
    ])
    print('Downloaded!')

subprocess.check_call(['tar', 'xzf', str(TAR), '-C', str(ROOT)])
print('Extracted to', ROOT)
!find {ROOT}/Data -type f | head -10

In [ ]:
# Cell 3: Install deps
!pip install -q numpy torch torchvision scikit-learn
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 4: All model code (self-contained, no repo clone needed)
import json
import logging
import time
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('v3_train')

V3_CLASSES = ['LDA', 'LVF', 'CCF', 'OTHER']
ROOT = Path('/content/marslab_v3')


@dataclass
class TileClassifierConfig:
    embed_dim: int = 768
    mola_dim: int = 25
    hidden_dim: int = 256
    num_classes: int = 4
    dropout: float = 0.3
    lr: float = 1e-4
    weight_decay: float = 1e-4
    epochs: int = 100
    patience: int = 15
    batch_size: int = 512
    focal_gamma: float = 1.5
    label_smoothing: float = 0.1
    confident_threshold: float = 0.6
    mixed_threshold: float = 0.2
    mixed_loss_weight: float = 0.5
    other_subsample_ratio: float = 0.3
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15


class TileLandformClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        input_dim = config.embed_dim + config.mola_dim
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, config.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.num_classes),
        )

    def forward(self, embeddings, mola):
        return self.classifier(torch.cat([embeddings, mola], dim=1))


class FocalLoss(nn.Module):
    def __init__(self, gamma=1.5, label_smoothing=0.1, weight=None):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.class_weight = weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.class_weight,
                             label_smoothing=self.label_smoothing, reduction='none')
        probs = F.softmax(logits, dim=1)
        target_probs = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1.0 - target_probs) ** self.gamma
        return (focal_weight * ce).mean()


class TileLabelDataset(Dataset):
    def __init__(self, tile_labels, tile_indices, config, is_train=True, embeddings_by_image=None):
        self.config = config
        self.is_train = is_train
        self.embeddings_by_image = embeddings_by_image
        self.samples = []
        other_samples = []
        self.class_to_idx = {cls: i for i, cls in enumerate(V3_CLASSES)}

        for idx in tile_indices:
            t = tile_labels[idx]
            label = t.get('label')
            label_type = t.get('label_type', '')
            if label == 'UNLABELED' or label is None:
                continue
            img_id = t['image_id']
            if embeddings_by_image is not None and img_id not in embeddings_by_image:
                continue
            sample = {
                'image_id': img_id, 'tile_idx': t['tile_idx'],
                'label': label, 'label_type': label_type,
                'coverage': t.get('coverage', {}),
            }
            if label == 'OTHER':
                other_samples.append(sample)
            else:
                self.samples.append(sample)

        if is_train and other_samples:
            import random
            n_keep = max(1, int(len(other_samples) * config.other_subsample_ratio))
            random.shuffle(other_samples)
            other_samples = other_samples[:n_keep]
        self.samples.extend(other_samples)
        logger.info('TileLabelDataset: %d samples (%s)', len(self.samples),
                     dict(Counter(s['label'] for s in self.samples)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        all_embeddings = self.embeddings_by_image[sample['image_id']]
        tile_idx = sample['tile_idx']
        if tile_idx < len(all_embeddings):
            embedding = all_embeddings[tile_idx]
        else:
            embedding = np.zeros((self.config.embed_dim,), dtype=np.float32)
        mola = np.zeros((self.config.mola_dim,), dtype=np.float32)
        label_idx = self.class_to_idx.get(sample['label'], self.class_to_idx['OTHER'])
        result = {
            'embedding': torch.from_numpy(embedding).float(),
            'mola': torch.from_numpy(mola).float(),
            'label': torch.tensor(label_idx, dtype=torch.long),
            'label_type': sample['label_type'],
        }
        if sample['label_type'] == 'mixed':
            coverage = sample.get('coverage', {})
            soft_target = torch.zeros(self.config.num_classes, dtype=torch.float32)
            for cls, frac in coverage.items():
                if cls in self.class_to_idx:
                    soft_target[self.class_to_idx[cls]] = frac
            remaining = 1.0 - soft_target.sum()
            if remaining > 0:
                soft_target[self.class_to_idx['OTHER']] = remaining
            result['soft_target'] = soft_target
        return result

    def get_class_weights(self):
        counts = Counter(s['label'] for s in self.samples)
        total = len(self.samples)
        weights = torch.zeros(len(V3_CLASSES))
        for cls, idx in self.class_to_idx.items():
            cnt = counts.get(cls, 1)
            weights[idx] = total / (len(V3_CLASSES) * cnt)
        return weights

    def get_sample_weights(self):
        class_weights = self.get_class_weights()
        return torch.tensor([float(class_weights[self.class_to_idx.get(s['label'], 3)]) for s in self.samples])


def _collate(batch):
    result = {
        'embedding': torch.stack([b['embedding'] for b in batch]),
        'mola': torch.stack([b['mola'] for b in batch]),
        'label': torch.stack([b['label'] for b in batch]),
    }
    if any('soft_target' in b for b in batch):
        soft, has_soft = [], []
        for b in batch:
            if 'soft_target' in b:
                soft.append(b['soft_target'])
                has_soft.append(True)
            else:
                soft.append(torch.zeros(4))
                has_soft.append(False)
        result['soft_target'] = torch.stack(soft)
        result['has_soft'] = torch.tensor(has_soft, dtype=torch.bool)
    return result


print('Model code loaded.')

In [ ]:
# Cell 5: Load data + build datasets
LABELS_PATH = ROOT / 'Data/HiRISE/v3_output/tile_labels_v3.json'
SPLITS_PATH = ROOT / 'Data/HiRISE/v3_output/tile_splits_v3.json'
EMB_PATH = ROOT / 'Data/HiRISE/v2_output/embeddings_ssl/embeddings_by_image.npy'

with open(LABELS_PATH) as f:
    tile_labels = json.load(f)
with open(SPLITS_PATH) as f:
    splits = json.load(f)

emb = np.load(str(EMB_PATH), allow_pickle=True).item()
print(f'Loaded {len(tile_labels)} tile labels, {len(emb)} image embeddings')
print(f'Train: {len(splits["train"])}, Val: {len(splits["val"])}, Test: {len(splits["test"])}')

cfg = TileClassifierConfig(epochs=100, batch_size=512, lr=1e-4, patience=15)
train_ds = TileLabelDataset(tile_labels, splits['train'], cfg, True, emb)
val_ds = TileLabelDataset(tile_labels, splits['val'], cfg, False, emb)
test_ds = TileLabelDataset(tile_labels, splits['test'], cfg, False, emb)

In [ ]:
# Cell 6: TRAIN
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

model = TileLandformClassifier(cfg).to(device)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
class_weights = train_ds.get_class_weights().to(device)
criterion = FocalLoss(gamma=cfg.focal_gamma, label_smoothing=cfg.label_smoothing, weight=class_weights)

train_sampler = WeightedRandomSampler(weights=train_ds.get_sample_weights().tolist(),
                                       num_samples=len(train_ds), replacement=True)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=train_sampler,
                          num_workers=2, pin_memory=True, collate_fn=_collate)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size * 2, shuffle=False,
                        num_workers=2, pin_memory=True, collate_fn=_collate)

total_steps = cfg.epochs * (len(train_ds) // cfg.batch_size + 1)
warmup_steps = int(total_steps * 0.1)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=cfg.lr, total_steps=total_steps,
                                                  pct_start=warmup_steps / total_steps, anneal_strategy='cos')

from sklearn.metrics import f1_score, accuracy_score

best_f1 = 0.0
best_epoch = -1
patience_counter = 0
SAVE_PATH = ROOT / 'best_tile_classifier.pt'

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()

    model.train()
    train_loss_sum = 0.0
    n_train = 0
    for batch in train_loader:
        emb_b = batch['embedding'].to(device)
        mola_b = batch['mola'].to(device)
        labels_b = batch['label'].to(device)
        logits = model(emb_b, mola_b)
        loss = criterion(logits, labels_b)
        if 'soft_target' in batch and 'has_soft' in batch:
            soft_t = batch['soft_target'].to(device)
            has_s = batch['has_soft'].to(device)
            if has_s.any():
                log_probs = F.log_softmax(logits[has_s], dim=1)
                kl = F.kl_div(log_probs, soft_t[has_s], reduction='batchmean')
                loss = loss + cfg.mixed_loss_weight * kl
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss_sum += loss.item()
        n_train += 1

    model.eval()
    all_preds, all_labels = [], []
    val_loss_sum = 0.0
    n_val = 0
    with torch.no_grad():
        for batch in val_loader:
            emb_b = batch['embedding'].to(device)
            mola_b = batch['mola'].to(device)
            labels_b = batch['label'].to(device)
            logits = model(emb_b, mola_b)
            val_loss_sum += criterion(logits, labels_b).item()
            n_val += 1
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels_b.cpu().tolist())

    lf_mask = [i for i, y in enumerate(all_labels) if y < 3]
    if lf_mask:
        lf_f1 = f1_score([all_labels[i] for i in lf_mask], [all_preds[i] for i in lf_mask], average='macro', zero_division=0)
    else:
        lf_f1 = 0.0
    acc = accuracy_score(all_labels, all_preds)
    elapsed = time.time() - t0

    print(f'Epoch {epoch:3d}/{cfg.epochs}  train_loss={train_loss_sum/max(n_train,1):.4f}  '
          f'val_loss={val_loss_sum/max(n_val,1):.4f}  lf_F1={lf_f1:.4f}  acc={acc:.4f}  ({elapsed:.1f}s)')

    if lf_f1 > best_f1:
        best_f1 = lf_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save({'model_state_dict': model.state_dict(), 'config': asdict(cfg),
                     'epoch': epoch, 'best_landform_macro_f1': lf_f1, 'classes': V3_CLASSES}, SAVE_PATH)
        print(f'  >> Saved best checkpoint (F1={lf_f1:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= cfg.patience:
            print(f'Early stopping at epoch {epoch} (patience={cfg.patience})')
            break

print(f'\nTraining complete. Best landform F1={best_f1:.4f} at epoch {best_epoch}')
print(f'Checkpoint: {SAVE_PATH}')

In [ ]:
# Cell 7: Evaluate on val + test
ckpt = torch.load(SAVE_PATH, map_location='cpu')
eval_model = TileLandformClassifier(cfg)
eval_model.load_state_dict(ckpt['model_state_dict'])
eval_model.eval()

def evaluate_split(ds, name):
    yt, yp = [], []
    with torch.no_grad():
        for i in range(len(ds)):
            s = ds[i]
            logits = eval_model(s['embedding'].unsqueeze(0), s['mola'].unsqueeze(0))
            yp.append(int(logits.argmax(dim=1).item()))
            yt.append(int(s['label'].item()))
    overall = f1_score(yt, yp, average='macro', zero_division=0)
    lf_idx = [i for i, y in enumerate(yt) if y < 3]
    lf_f1 = f1_score([yt[i] for i in lf_idx], [yp[i] for i in lf_idx], average='macro', zero_division=0) if lf_idx else 0.0
    per = {}
    for c, cls in enumerate(V3_CLASSES):
        per[cls] = round(f1_score([1 if y==c else 0 for y in yt], [1 if p==c else 0 for p in yp], zero_division=0), 4)
    print(f'\n=== {name} ({len(ds)} samples) ===')
    print(f'  Overall macro-F1: {overall:.4f}')
    print(f'  Landform macro-F1 (LDA/LVF/CCF): {lf_f1:.4f}')
    for cls, f in per.items():
        print(f'    {cls}: {f}')
    return {'samples': len(ds), 'overall_f1': overall, 'landform_f1': lf_f1, 'per_class': per}

val_result = evaluate_split(val_ds, 'Validation')
test_result = evaluate_split(test_ds, 'Test')

In [ ]:
# Cell 8: Copy checkpoint to Google Drive
from google.colab import drive
import shutil

drive.mount('/content/drive')
drive_dir = Path('/content/drive/MyDrive/MarsLab_V3')
drive_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(SAVE_PATH, drive_dir / 'best_tile_classifier.pt')

summary = {'best_f1': best_f1, 'best_epoch': best_epoch, 'val': val_result, 'test': test_result}
with open(drive_dir / 'training_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Saved to Google Drive: {drive_dir}')
print(f'  best_tile_classifier.pt')
print(f'  training_results.json')